# Laboratorio AI SQL - A2A Energia
Esplorazione delle funzioni Cortex AI SQL sui dati energetici A2A.

**Funzioni testate:**
- SENTIMENT - Analisi del sentiment
- SUMMARIZE - Riassunto automatico
- TRANSLATE - Traduzione multilingua
- COMPLETE - Prompt LLM generico
- CLASSIFY - Classificazione automatica
- EXTRACT - Estrazione entita'

In [ ]:
%%sql -r dataframe_1
USE DATABASE POWERUTILITY;
USE SCHEMA PUBLIC;
USE WAREHOUSE COMPUTE_WH;

## 1. Sentiment Analysis
Analizziamo il sentiment dei feedback clienti per identificare clienti insoddisfatti e aree di miglioramento.

In [ ]:
--- Sentiment analysis su tutti i feedback (deduplicati)
WITH feedback_unici AS (
    SELECT DISTINCT TESTO_FEEDBACK, CANALE, CATEGORIA
    FROM FEEDBACK_CLIENTI
)
SELECT
    CANALE,
    CATEGORIA,
    SUBSTR(TESTO_FEEDBACK, 1, 80) || '...' AS TESTO_BREVE,
    ROUND(SNOWFLAKE.CORTEX.SENTIMENT(TESTO_FEEDBACK), 3) AS SENTIMENT_SCORE,
    CASE
        WHEN SNOWFLAKE.CORTEX.SENTIMENT(TESTO_FEEDBACK) > 0.3 THEN 'POSITIVO'
        WHEN SNOWFLAKE.CORTEX.SENTIMENT(TESTO_FEEDBACK) < -0.3 THEN 'NEGATIVO'
        ELSE 'NEUTRO'
    END AS SENTIMENT_LABEL
FROM feedback_unici
ORDER BY SENTIMENT_SCORE ASC
LIMIT 20;

In [ ]:
%%sql -r dataframe_3
--- Statistiche sentiment aggregate per categoria
SELECT
    CATEGORIA,
    COUNT(*) AS NUM_FEEDBACK,
    ROUND(AVG(SNOWFLAKE.CORTEX.SENTIMENT(TESTO_FEEDBACK)), 3) AS AVG_SENTIMENT,
    ROUND(MIN(SNOWFLAKE.CORTEX.SENTIMENT(TESTO_FEEDBACK)), 3) AS MIN_SENTIMENT,
    ROUND(MAX(SNOWFLAKE.CORTEX.SENTIMENT(TESTO_FEEDBACK)), 3) AS MAX_SENTIMENT,
    SUM(CASE WHEN SNOWFLAKE.CORTEX.SENTIMENT(TESTO_FEEDBACK) < -0.3 THEN 1 ELSE 0 END) AS NEGATIVI,
    SUM(CASE WHEN SNOWFLAKE.CORTEX.SENTIMENT(TESTO_FEEDBACK) > 0.3 THEN 1 ELSE 0 END) AS POSITIVI
FROM FEEDBACK_CLIENTI
GROUP BY CATEGORIA
ORDER BY AVG_SENTIMENT ASC;

## 2. Summarization
Generiamo riassunti automatici dei documenti tecnici per una rapida consultazione.

In [ ]:
--- Riassunto automatico dei documenti tecnici con traduzione in italiano
SELECT
    TITOLO,
    TIPO_DOCUMENTO,
    AREA,
    SNOWFLAKE.CORTEX.SUMMARIZE(CONTENUTO) AS RIASSUNTO,
    SNOWFLAKE.CORTEX.TRANSLATE(SNOWFLAKE.CORTEX.SUMMARIZE(CONTENUTO), 'en', 'it') AS RIASSUNTO_IT
FROM DOCUMENTI_TECNICI
WHERE TIPO_DOCUMENTO IN ('REPORT', 'PROCEDURA')
LIMIT 5;

## 3. Traduzione Multilingua
Traduciamo i feedback dei clienti in inglese per report internazionali.

In [ ]:
%%sql -r dataframe_5
--- Traduzione feedback in inglese e tedesco
SELECT
    FEEDBACK_ID,
    TESTO_FEEDBACK AS ORIGINALE_IT,
    SNOWFLAKE.CORTEX.TRANSLATE(TESTO_FEEDBACK, 'it', 'en') AS TRADUZIONE_EN,
    SNOWFLAKE.CORTEX.TRANSLATE(TESTO_FEEDBACK, 'it', 'de') AS TRADUZIONE_DE
FROM FEEDBACK_CLIENTI
LIMIT 5;

## 4. AI Complete - Prompt Generici
Usiamo il modello LLM per analisi avanzate: categorizzazione, suggerimento azioni, generazione report.

In [ ]:
%%sql -r dataframe_6
--- Analisi del feedback con suggerimento di azione
SELECT
    FEEDBACK_ID,
    CATEGORIA,
    TESTO_FEEDBACK,
    SNOWFLAKE.CORTEX.COMPLETE(
        'mistral-large2',
        CONCAT(
            'Sei un esperto di customer care nel settore energetico italiano. ',
            'Analizza il seguente feedback di un cliente A2A e fornisci: ',
            '1) Classificazione urgenza (CRITICA/ALTA/MEDIA/BASSA) ',
            '2) Azione suggerita in una frase. ',
            'Feedback: ', TESTO_FEEDBACK
        )
    ) AS ANALISI_AI
FROM FEEDBACK_CLIENTI
WHERE PRIORITA = 'ALTA'
LIMIT 5;

In [ ]:
%%sql -r dataframe_7
--- Generazione automatica di un mini-report con AI_COMPLETE
WITH stats AS (
    SELECT
        ROUND(SUM(CONSUMO_KWH), 0) AS TOT_KWH,
        ROUND(SUM(CONSUMO_SMC), 0) AS TOT_SMC,
        ROUND(SUM(COSTO_TOTALE_EUR), 0) AS TOT_COSTO,
        ROUND(AVG(COSTO_TOTALE_EUR), 2) AS AVG_COSTO,
        COUNT(DISTINCT CLIENTE_ID) AS NUM_CLIENTI
    FROM CONSUMI_ENERGIA
    WHERE MESE >= DATEADD('month', -3, CURRENT_DATE())
)
SELECT SNOWFLAKE.CORTEX.COMPLETE(
    'mistral-large2',
    CONCAT(
        'Genera un breve report in italiano (max 200 parole) sui consumi energetici A2A degli ultimi 3 mesi. ',
        'Dati: Consumo totale elettricita'' = ', TOT_KWH::VARCHAR, ' kWh, ',
        'Consumo totale gas = ', TOT_SMC::VARCHAR, ' smc, ',
        'Costo totale = ', TOT_COSTO::VARCHAR, ' EUR, ',
        'Costo medio per cliente = ', AVG_COSTO::VARCHAR, ' EUR, ',
        'Clienti attivi = ', NUM_CLIENTI::VARCHAR, '. ',
        'Includi trend e raccomandazioni.'
    )
) AS REPORT_ENERGETICO
FROM stats;

## 5. AI Classify - Classificazione Automatica
Classifichiamo automaticamente i feedback in categorie di urgenza.

In [ ]:
%%sql -r dataframe_8
--- Classificazione automatica dei feedback
SELECT
    FEEDBACK_ID,
    TESTO_FEEDBACK,
    SNOWFLAKE.CORTEX.CLASSIFY_TEXT(
        TESTO_FEEDBACK,
        ['Problema tecnico urgente', 'Richiesta informazioni', 'Reclamo commerciale', 'Complimento', 'Richiesta modifica contratto']
    ):label::VARCHAR AS CLASSIFICAZIONE,
    ROUND(SNOWFLAKE.CORTEX.CLASSIFY_TEXT(
        TESTO_FEEDBACK,
        ['Problema tecnico urgente', 'Richiesta informazioni', 'Reclamo commerciale', 'Complimento', 'Richiesta modifica contratto']
    ):score::FLOAT, 3) AS CONFIDENCE
FROM FEEDBACK_CLIENTI
LIMIT 15;

## 6. AI Extract - Estrazione Entita'
Estraiamo informazioni strutturate dai testi dei feedback.

In [ ]:
%%sql -r dataframe_9
--- Estrazione entita' dai feedback
SELECT
    FEEDBACK_ID,
    TESTO_FEEDBACK,
    SNOWFLAKE.CORTEX.EXTRACT_ANSWER(
        TESTO_FEEDBACK,
        'Qual e'' il problema principale segnalato dal cliente?'
    ) AS PROBLEMA_PRINCIPALE,
    SNOWFLAKE.CORTEX.EXTRACT_ANSWER(
        TESTO_FEEDBACK,
        'Il cliente menziona una localita'' specifica?'
    ) AS LOCALITA_MENZIONATA
FROM FEEDBACK_CLIENTI
WHERE CATEGORIA = 'GUASTI'
LIMIT 10;

## Conclusioni

In questo laboratorio abbiamo visto come le funzioni AI SQL di Snowflake permettono di:

- **Analizzare il sentiment** dei feedback clienti per identificare criticita'
- **Riassumere automaticamente** documenti tecnici complessi
- **Tradurre** contenuti per report multilingua
- **Generare analisi** e report con prompt LLM personalizzati
- **Classificare** automaticamente i feedback in categorie
- **Estrarre** informazioni strutturate da testi non strutturati

Tutto questo senza mai spostare i dati fuori da Snowflake e con la governance RBAC integrata.